<a href="https://colab.research.google.com/github/Lindasofiana07/ADVANCE_FE1/blob/master/Deeplearingtugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

train_df = pd.read_csv("train_multiwoz_intent.csv")
test_df = pd.read_csv("test_multiwoz_intent.csv")

print(train_df.shape)
print(test_df.shape)

print(train_df.head())
print(test_df.head())

(2716, 4)
(679, 4)
                                                text           intent  \
0             i want to make a booking for 6 people.       book_train   
1                    does that place have free wifi?       find_hotel   
2                                 i leave on monday.       find_train   
3  thank you so much. i don't think i need anythi...  book_restaurant   
4  that sounds like what i am looking for. yes, i...       find_hotel   

       domain  label  
0       train      2  
1       hotel      5  
2       train      9  
3  restaurant      1  
4       hotel      5  
                                                text           intent  \
0  i am looking for a place to stay. the hotel sh...       find_hotel   
1  i am looking for a train to cambridge. i would...       find_train   
2  fitzwilliam museum sounds great. what is the p...  find_attraction   
3  would you be able to suggest a steakhouse rest...  find_restaurant   
4  great, thanks for that information o

In [2]:
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

In [3]:
print(sorted(train_df["label"].unique()))
print(sorted(test_df["label"].unique()))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [4]:
model_names = {
    "RoBERTa": "sentence-transformers/all-roberta-large-v1",
    "DistilRoBERTa": "sentence-transformers/all-distilroberta-v1",
    "MPNet": "sentence-transformers/all-mpnet-base-v2",
    "MiniLM-L6": "sentence-transformers/all-MiniLM-L6-v2",
    "MiniLM-L12": "sentence-transformers/all-MiniLM-L12-v2"
}

In [5]:
import tensorflow as tf

def build_classifier(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import time

single_results = {}
hard_predictions = []
soft_predictions = []

In [7]:
for name, model_path in model_names.items():

    print(f"\nTraining {name}")

    st_model = SentenceTransformer(model_path)

    X_train_embed = st_model.encode(
        X_train.tolist(),
        show_progress_bar=True
    )

    X_test_embed = st_model.encode(
        X_test.tolist(),
        show_progress_bar=True
    )

    classifier = build_classifier(
        X_train_embed.shape[1],
        len(train_df["label"].unique())
    )

    classifier.fit(
        X_train_embed,
        y_train,
        epochs=20,
        batch_size=32
    )

    probs = classifier.predict(X_test_embed)

    preds = np.argmax(probs, axis=1)

    hard_predictions.append(preds)
    soft_predictions.append(probs)

    single_results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, average="weighted"),
        "Recall": recall_score(y_test, preds, average="weighted"),
        "F1": f1_score(y_test, preds, average="weighted")
    }


Training RoBERTa


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 1/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.4823 - loss: 1.8019
Epoch 2/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6528 - loss: 1.2203
Epoch 3/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6852 - loss: 1.0739
Epoch 4/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6937 - loss: 1.0191
Epoch 5/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7132 - loss: 0.9703
Epoch 6/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7176 - loss: 0.9379
Epoch 7/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7301 - loss: 0.9132
Epoch 8/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7246 - loss: 0.8957
Epoch 9/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7375 - loss: 0.8745
Epoch 10/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7382 - loss: 0.8651
Epoch 11/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7489 - loss: 0.8454
Epoch 12/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7504 - l